In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
import xesmf as xe

In [2]:
ROOT_DIR = Path.home() / "EASM"
DATA_DIR = Path("D:/Data/NCAR")
PROCESSED_DIR = ROOT_DIR / "data"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
files = sorted(DATA_DIR.glob("sst.day.mean.????.nc"))

In [4]:
output_file = PROCESSED_DIR / "oisst_regrid_jja_1982_2025.nc"

In [5]:
ds = xr.open_mfdataset(files, engine="h5netcdf")

In [6]:
ds = ds.sel(time=slice("1982", "2025"))

In [7]:
ds = ds.sel(time=ds.time.dt.month.isin([6, 7, 8]))

In [8]:
ds_out = xr.Dataset(
    {
        "lat": (["lat"], np.arange(-90, 90.1, 2.5), {"units": "degrees_north"}),
        "lon": (["lon"], np.arange(0, 360.0, 2.5), {"units": "degrees_east"}),
    }
)
ds_out

<xarray.Dataset> Size: 2kB
Dimensions:  (lat: 73, lon: 144)
Coordinates:
  * lat      (lat) float64 584B -90.0 -87.5 -85.0 -82.5 ... 82.5 85.0 87.5 90.0
  * lon      (lon) float64 1kB 0.0 2.5 5.0 7.5 10.0 ... 350.0 352.5 355.0 357.5
Data variables:
    *empty*

In [9]:
regridder = xe.Regridder(ds, ds_out, "conservative")
regridder

c:\Users\user\miniforge3\envs\myenv\Lib\site-packages\xesmf\backend.py:56: UserWarning: Latitude is outside of [-90, 90]
  warnings.warn('Latitude is outside of [-90, 90]')


xESMF Regridder 
Regridding algorithm:       conservative 
Weight filename:            conservative_720x1440_73x144.nc 
Reuse pre-computed weights? False 
Input grid shape:           (720, 1440) 
Output grid shape:          (73, 144) 
Periodic in longitude?      False

In [10]:
dr_out = regridder(ds, keep_attrs=True)
dr_out = dr_out.compute()

In [11]:
ds.close()

In [12]:
dr_out.to_netcdf(output_file)